# PVGIS catalog explorer (offline)

Inspect the bundled product catalog without touching the network: which tools
ship, the REST endpoint behind each, and the canonical value columns each
returns. This is the no-network companion to the live quickstart.

In [1]:
import pandas as pd

from earthlens.pvgis import Catalog

## The shipped products

`Catalog()` loads the bundled `pvgis_data_catalog.yaml`. Each row is a frozen
pydantic `Product` carrying the tool, endpoint, default params, and columns.

In [2]:
catalog = Catalog()
print(catalog.available())

rows = [
    {
        "id": pid,
        "tool": catalog.get(pid).tool,
        "endpoint": catalog.get(pid).endpoint,
        "n_columns": len(catalog.get(pid).columns),
        "description": catalog.get(pid).description.split(".")[0],
    }
    for pid in catalog.available()
]
pd.DataFrame(rows)

['seriescalc', 'tmy']


,id,tool,endpoint,n_columns,description
0,seriescalc,seriescalc,seriescalc,6,Hourly solar-radiation time series (global in-...
1,tmy,tmy,tmy,10,Typical Meteorological Year — an hourly multi-...


## A product's columns

`get(<id>)` returns the row; `columns` lists the canonical value columns the
parser keeps (the `time` column plus the per-tool variables).

In [3]:
print("seriescalc:", catalog.get("seriescalc").columns)
print("tmy:       ", catalog.get("tmy").columns)

seriescalc: ['time', 'G(i)', 'H_sun', 'T2m', 'WS10m', 'Int']
tmy:        ['time', 'T2m', 'RH', 'G(h)', 'Gb(n)', 'Gd(h)', 'IR(h)', 'WS10m', 'WD10m', 'SP']


## Did-you-mean on a typo

An unknown id raises a `ValueError` with the closest match.

In [4]:
try:
    catalog.get("seriescal")
except ValueError as exc:
    print(exc)

'seriescal' is not in the PVGIS product catalog. Known products: ['seriescalc', 'tmy']. Did you mean 'seriescalc'?


## Takeaway

The catalog is the single source of truth for which PVGIS tools the backend
exposes and the columns each returns — query it offline before issuing any
request.